# Geo Coding

In [14]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv('deduplicated_outlets_corrected.csv')
df.head()

,name,outlet_name,address,postal_code,latitude,longitude,muis_halal_status,cuisine,website,phone,certificate_number,type,scheme,sub_scheme,data_sources,muis_place_id,halaltag_place_id
0,ARTEASE CAFE,NaN,1 PUNGGOL DRIVE #02-45 ONE PUNGGOL 828629,828629.0,NaN,NaN,MUIS Halal Certified,NaN,NaN,NaN,EERT20230001273,0.0,100.0,106.0,"muis,halaltag",d3fac8ea-3845-4de7-bb71-c4a6aebaf302,2222.0
1,ARTEASE CAFE,NaN,900 SOUTH WOODLANDS DRIVE #01-03 WOODLANDS CIV...,730900.0,NaN,NaN,MUIS Halal Certified,NaN,NaN,NaN,EERT20230001274,0.0,100.0,106.0,"muis,halaltag",d3fac8ea-3845-4de7-bb71-c4a6aebaf302,2222.0
2,Atrium Kitchen @ Holiday Inn Singapore Atrium,NaN,317 OUTRAM ROAD #04-00 HOLIDAY INN SINGAPORE A...,169075.0,NaN,NaN,MUIS Halal Certified,NaN,NaN,NaN,FPCN21120001634,0.0,200.0,201.0,"muis,halaltag",5d05fdd7-d64f-464c-a0a9-e9d511d0efb5,1236.0
3,Adam Chicken,NaN,503 WEST COAST DRIVE #01-28 Ayer Rajah Food Ce...,120503.0,NaN,NaN,MUIS Halal Certified,NaN,NaN,NaN,EEHN22020013198,0.0,100.0,103.0,"muis,halaltag",a38ed3f3-c19b-4c8c-9a5f-858750d8c7af,1455.0
4,Al Zamira Restaurant,NaN,724 ANG MO KIO AVENUE 6 #01-04 560724,560724.0,NaN,NaN,MUIS Halal Certified,NaN,NaN,NaN,EEHN21050012316,0.0,100.0,103.0,"muis,halaltag",0b0eb390-cdb8-4f3b-8bc5-d45cc75e3072,2394.0


Filtering out data with these specifications

1. Scheme = 200
2. Subscheme = 201, 202

In [6]:
df = df[df['scheme'] != 200 & ~df['sub_scheme'].isin([201, 202])]
df.drop(columns=['cuisine', 'website', 'phone', 'latitude', 'longitude'], inplace=True)
df.head()

,name,outlet_name,address,postal_code,muis_halal_status,certificate_number,type,scheme,sub_scheme,data_sources,muis_place_id,halaltag_place_id
0,ARTEASE CAFE,NaN,1 PUNGGOL DRIVE #02-45 ONE PUNGGOL 828629,828629.0,MUIS Halal Certified,EERT20230001273,0.0,100.0,106.0,"muis,halaltag",d3fac8ea-3845-4de7-bb71-c4a6aebaf302,2222.0
1,ARTEASE CAFE,NaN,900 SOUTH WOODLANDS DRIVE #01-03 WOODLANDS CIV...,730900.0,MUIS Halal Certified,EERT20230001274,0.0,100.0,106.0,"muis,halaltag",d3fac8ea-3845-4de7-bb71-c4a6aebaf302,2222.0
2,Atrium Kitchen @ Holiday Inn Singapore Atrium,NaN,317 OUTRAM ROAD #04-00 HOLIDAY INN SINGAPORE A...,169075.0,MUIS Halal Certified,FPCN21120001634,0.0,200.0,201.0,"muis,halaltag",5d05fdd7-d64f-464c-a0a9-e9d511d0efb5,1236.0
3,Adam Chicken,NaN,503 WEST COAST DRIVE #01-28 Ayer Rajah Food Ce...,120503.0,MUIS Halal Certified,EEHN22020013198,0.0,100.0,103.0,"muis,halaltag",a38ed3f3-c19b-4c8c-9a5f-858750d8c7af,1455.0
4,Al Zamira Restaurant,NaN,724 ANG MO KIO AVENUE 6 #01-04 560724,560724.0,MUIS Halal Certified,EEHN21050012316,0.0,100.0,103.0,"muis,halaltag",0b0eb390-cdb8-4f3b-8bc5-d45cc75e3072,2394.0


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6897 entries, 0 to 6896
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   name                6892 non-null   object 
 1   outlet_name         2131 non-null   object 
 2   address             6726 non-null   object 
 3   postal_code         6576 non-null   float64
 4   muis_halal_status   6897 non-null   object 
 5   certificate_number  4550 non-null   object 
 6   type                4550 non-null   float64
 7   scheme              4550 non-null   float64
 8   sub_scheme          4550 non-null   float64
 9   data_sources        6897 non-null   object 
 10  muis_place_id       4550 non-null   object 
 11  halaltag_place_id   4151 non-null   float64
dtypes: float64(5), object(7)
memory usage: 646.7+ KB


In [8]:
post = pd.read_csv('building2025_PC.csv', index_col = 0)
post.head()

,ADDRESS,BLK_NO,BUILDING,POSTAL,ROAD_NAME,X,Y
1,101A BAYFRONT AVENUE TEMPORARY SITE OFFICE SIN...,101A,TEMPORARY SITE OFFICE,18895,BAYFRONT AVENUE,30485.511362,28685.612665
2,1 STRAITS BOULEVARD SINGAPORE CHINESE CULTURAL...,1,SINGAPORE CHINESE CULTURAL CENTRE,18906,STRAITS BOULEVARD,29809.365407,28700.236127
3,11A STRAITS BOULEVARD TEMPORARY SITE OFFICE SI...,11A,TEMPORARY SITE OFFICE,18907,STRAITS BOULEVARD,30041.838898,28602.987244
4,2 CENTRAL BOULEVARD IOI CENTRAL BOULEVARD TOWE...,2,IOI CENTRAL BOULEVARD TOWERS,18916,CENTRAL BOULEVARD,30024.919852,29136.807026
5,21 PARK STREET MARINA BAY MRT STATION SINGAPOR...,21,MARINA BAY MRT STATION,18925,PARK STREET,30369.006866,28753.531902


In [10]:
def postal_normalize(postal):
    if pd.isna(postal):
        return None
    try:
        return str(int(postal)).zfill(6)
    except:
        return None
    
post['POSTAL'] = post['POSTAL'].apply(postal_normalize)
df['postal_code'] = df['postal_code'].apply(postal_normalize)

In [11]:


# ============================================================================
# STEP 2: Primary match - POSTAL CODE (exact)
# ============================================================================
print(f"\nSTEP 2: Matching by postal code...")

# Create lookup dict for faster matching (postal -> first matching building)
postal_lookup = {}
for idx, row in post.iterrows():
    postal = row['POSTAL']
    if postal and postal not in postal_lookup:
        postal_lookup[postal] = {
            'X': row['X'],
            'Y': row['Y'],
            'ADDRESS': row['ADDRESS'],
            'BUILDING': row['BUILDING'],
            'ROAD_NAME': row['ROAD_NAME']
        }

# Match by postal
matched_postal = []
for idx, row in df.iterrows():
    postal = row['postal_code']
    if postal and postal in postal_lookup:
        matched_postal.append({
            'idx': idx,
            'X': postal_lookup[postal]['X'],
            'Y': postal_lookup[postal]['Y'],
            'match_type': 'postal_exact'
        })

print(f"  Matched by postal: {len(matched_postal):,}")



STEP 2: Matching by postal code...
  Matched by postal: 6,539


In [12]:

# ============================================================================
# STEP 3: Fallback match - ADDRESS/BUILDING/ROAD fuzzy matching
# ============================================================================
print(f"\nSTEP 3: Fallback matching for unmatched records...")

def normalize_text(text):
    """Normalize text for comparison"""
    if pd.isna(text):
        return ""
    return str(text).upper().strip()

matched_postal_set = {m['idx'] for m in matched_postal}
unmatched_indices = [i for i in range(len(df)) if i not in matched_postal_set]

print(f"  Unmatched records to process: {len(unmatched_indices):,}")






STEP 3: Fallback matching for unmatched records...
  Unmatched records to process: 358


In [13]:
matched_fuzzy = []
for idx in unmatched_indices[:500]:  # Limit fuzzy matching to prevent long runtime
    row = df.iloc[idx]
    address = normalize_text(row.get('address', ''))
    
    if not address:
        continue
    
    # Try exact substring match on ADDRESS column
    building_matches = post[post['ADDRESS'].str.upper().str.contains(address[:50], na=False, regex=False)]
    
    if len(building_matches) > 0:
        first_match = building_matches.iloc[0]
        matched_fuzzy.append({
            'idx': idx,
            'X': first_match['X'],
            'Y': first_match['Y'],
            'match_type': 'address_fuzzy'
        })

print(f"  Matched by fuzzy address: {len(matched_fuzzy):,}")

  Matched by fuzzy address: 56


In [15]:
# ============================================================================
# STEP 4: Combine all matches and update dataframe
# ============================================================================
print(f"\nSTEP 4: Applying geocoding results...")

all_matches = matched_postal + matched_fuzzy
match_dict = {m['idx']: m for m in all_matches}

# Initialize new columns
df['X'] = np.nan
df['Y'] = np.nan
df['geocode_source'] = 'unmatched'

# Apply matches
for idx, match in match_dict.items():
    df.at[idx, 'X'] = match['X']
    df.at[idx, 'Y'] = match['Y']
    df.at[idx, 'geocode_source'] = match['match_type']



STEP 4: Applying geocoding results...


In [24]:
# ============================================================================
# STEP 5: VALIDATION & QA
# ============================================================================
print("\n" + "="*70)
print("GEOCODING QA SUMMARY")
print("="*70)

# Row count check
assert len(df) == 6897, f"ERROR: Row count changed! Expected 6897, got {len(df)}"
print(f"✓ Row count preserved: {len(df):,}")

# Match statistics
geocoded = df['X'].notna().sum()
print(f"\n✓ Geocoding coverage: {geocoded:,}/{len(df):,} ({100*geocoded/len(df):.1f}%)")

print(f"\nMatch breakdown:")
for source in ['postal_exact', 'address_fuzzy', 'unmatched']:
    count = (df['geocode_source'] == source).sum()
    pct = 100 * count / len(df)
    print(f"  {source:20s}: {count:5,} ({pct:5.1f}%)")

# Coordinate validation
print(f"\nCoordinate ranges (X, Y in SVY21 meters):")
print(f"  X (X):  {df['X'].min():.2f} to {df['X'].max():.2f}")
print(f"  Y (Y): {df['Y'].min():.2f} to {df['Y'].max():.2f}")

# Analyze unmatched records
unmatched_df = df[df['geocode_source'] == 'unmatched']
print(f"\nUnmatched records analysis:")
print(f"  Total unmatched: {len(unmatched_df):,}")
print(f"  Sample unmatched records:")
print(unmatched_df[['name', 'address', 'postal_code']].head(10).to_string(index=False)) 
unmatched_df.data_sources.value_counts()




GEOCODING QA SUMMARY
✓ Row count preserved: 6,897

✓ Geocoding coverage: 6,595/6,897 (95.6%)

Match breakdown:
  postal_exact        : 6,539 ( 94.8%)
  address_fuzzy       :    56 (  0.8%)
  unmatched           :   302 (  4.4%)

Coordinate ranges (X, Y in SVY21 meters):
  X (X):  3608.99 to 46249.72
  Y (Y): 24251.56 to 49992.30

Unmatched records analysis:
  Total unmatched: 302
  Sample unmatched records:
                            name                                                     address postal_code
                    Artease Cafe 9 Woodlands Ave 9, #01 (Cultural Centre),  Singapore 738984      738984
           ALaleem Eating Corner                                                         NaN        None
                Apam Balik Power                                                         NaN        None
               Aiman Seafood BBQ                                                         NaN        None
                       Asap & Co                 70 Telok Ayer 

data_sources
halaltag    295
muis          7
Name: count, dtype: int64

In [26]:
df.to_csv('deduplicated_outlets_geocoded.csv', index=False)